# Probe a trained LeWM on OGBCubeDR (RunPod)

Fits **classical linear and MLP probes** on the frozen representations of a
trained LeWM checkpoint and asks, target by target, what survived encoding.
`Run.md` §4 describes the digit decals as existing for exactly this; this
notebook covers those and 19 other labels.

Flow: config → apply → GPU check → install → download dataset + verify
checkpoint → pick epoch → extract frozen features (trained **and** untrained
control) → fit probes → read the results → optional epoch sweep.

**No renderer needed.** Unlike `plan_lewm_ogbcubedr.ipynb` this notebook never
steps MuJoCo, so the `apt-get` GL block and `MUJOCO_GL` do not apply.

### What makes it a clean experiment

- **The encoder is frozen and evaluated once.** Every probe, every target and
  every capacity rung reads the same cached feature matrix, so differences
  between them cannot come from the encoder.
- **Splits are by episode, not by frame.** Frames 5 env-steps apart in a
  400-step episode are near-duplicates; a frame-level split would leak the
  test set into the train set and inflate every number. (Training itself uses
  a clip-level `random_split` — fine for fitting a world model, wrong for
  measuring one.)
- **Three rungs per target**: constant-predictor baseline → linear probe →
  MLP probe. A score is only ever read as a *difference* against the rung
  below it.
- **Two negative controls**, because a probe that reports success on
  something the image does not contain is measuring the experiment, not the
  model:
  - an **untrained encoder** of the same architecture (`random_init`) — how
    much of the score is just the ViT prior plus probe capacity;
  - a **16×16 pixel** read-out — how much was already sitting in the raw
    frame.
- **Labels the frame does not contain** are probed on purpose (`joint_vel`,
  `target_block_pos`, `block_0_pos`, `target_block`). They should fail — and
  each fails for a different, documented reason, so which one *doesn't* tells
  you what went wrong. See `scripts/probe/targets.py`.


## 1. Config

Edit the values below. Defaults assume the same `/workspace` network volume
`train_lewm_ogbcubedr.ipynb` used, so the dataset and checkpoint are already
in place and nothing needs downloading.


In [ ]:
import os

# --- repo ---
REPO_ROOT = '/workspace/stable-worldmodel'          # ← edit if you cloned it elsewhere

# --- storage (network volume) ---
STABLEWM_HOME = '/workspace'                        # datasets/, checkpoints/ live directly here

# --- checkpoint to probe ---
OUTPUT_MODEL_NAME = 'lewm_q4_dr'                    # matches the training run name
POLICY_EPOCH = None                                 # ← int to pin an epoch; None = latest on this volume

# --- fallback: only used if the dataset is NOT already on this volume ---
HF_TOKEN = os.environ.get('HF_TOKEN', '')
HF_DATASET_REPO_ID = '<your-hf-username-or-org>/ogbench-cube-quadruple-domain-randomized-expert'  # ← edit me

# --- experiment scale ---
# Episodes are the unit of the split; windows are sampled inside them.
#
# **Do not under-sample episodes.** Every episode re-draws lighting, camera
# angle, cube colours, floor/wall materials and the backdrop, so episode-level
# appearance is the dominant direction of variation in the features. With ~100
# train episodes a linear probe can fit episode identity and every
# within-episode target reads ~0 on held-out episodes -- measured, not
# hypothetical. 1000 episodes is the smallest setting that gave a usable
# signal; raise it before raising WINDOWS_PER_EPISODE, which adds correlated
# samples rather than independent ones.
TRAIN_EPISODES = 1000
VAL_EPISODES = 150
TEST_EPISODES = 250
WINDOWS_PER_EPISODE = 20
SEED = 0

# --- window layout: MUST match the training run ---
HISTORY_SIZE = 3        # wm.history_size
NUM_PREDS = 1           # wm.num_preds
FRAMESKIP = 5           # data.dataset.frameskip
IMG_SIZE = 224

# --- compute ---
BATCH_SIZE = 128
NUM_WORKERS = 6
DTYPE = 'float32'       # 'bfloat16' matches training precision and is ~2x faster

# --- derived, don't edit ---
DATASET_NAME = 'ogbench/cube_quadruple_dr_expert.lance'
DATASET_DIR = os.path.join(STABLEWM_HOME, 'datasets', 'ogbench', 'cube_quadruple_dr_expert.lance')
CHECKPOINT_DIR = os.path.join(STABLEWM_HOME, 'checkpoints', OUTPUT_MODEL_NAME)
RESULTS_ROOT = os.path.join(STABLEWM_HOME, 'probing', OUTPUT_MODEL_NAME)


## 2. Apply config


In [ ]:
os.environ['STABLEWM_HOME'] = STABLEWM_HOME
os.environ['HF_TOKEN'] = HF_TOKEN

os.makedirs(STABLEWM_HOME, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)

os.chdir(REPO_ROOT)  # os.chdir (not `%cd`) so it's identical whether run fresh or after a kernel restart

print('cwd            =', os.getcwd())
print('STABLEWM_HOME  =', os.environ['STABLEWM_HOME'])
print('DATASET_DIR    =', DATASET_DIR)
print('CHECKPOINT_DIR =', CHECKPOINT_DIR)
print('RESULTS_ROOT   =', RESULTS_ROOT)
!df -h /workspace


## 3. GPU check

Not strictly required — everything runs on CPU — but feature extraction is a
ViT-small forward over `(TRAIN+VAL+TEST episodes) x WINDOWS x (HISTORY+PREDS)`
frames, which is minutes on a GPU and hours on a CPU. Probe fitting itself is
seconds either way.


In [ ]:
import torch

print('torch', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU visible — extraction will run on CPU. Lower TRAIN_EPISODES / '
          'WINDOWS_PER_EPISODE, or set DTYPE = "bfloat16", if this is a test run.')


## 4. Install dependencies

Same scoped install as training (`Run.md` §3) — the probing code needs the
encoder and the Lance reader, nothing that renders.


In [ ]:
%pip install -q -e '.[train,format]' huggingface_hub


## 5. Verify (or fetch) the dataset, and verify the checkpoint


In [ ]:
if os.path.isdir(DATASET_DIR) and os.listdir(DATASET_DIR):
    print('Dataset already present:', DATASET_DIR)
else:
    assert HF_DATASET_REPO_ID and '<' not in HF_DATASET_REPO_ID, (
        'Dataset missing and HF_DATASET_REPO_ID is not set — edit the config cell.'
    )
    os.makedirs(DATASET_DIR, exist_ok=True)
    !hf download "$HF_DATASET_REPO_ID" --repo-type dataset --local-dir "$DATASET_DIR"


In [ ]:
import glob

assert os.path.isdir(CHECKPOINT_DIR), (
    f'{CHECKPOINT_DIR} does not exist. Copy the trained run onto this volume '
    '(e.g. from baseline_lewm_08_2026/checkpoints/) before probing.'
)
assert os.path.exists(os.path.join(CHECKPOINT_DIR, 'config.json')), (
    f'{CHECKPOINT_DIR} is missing config.json — the architecture is read from it, '
    'both for the trained model and for the random-init control.'
)
print(sorted(os.path.basename(p) for p in glob.glob(os.path.join(CHECKPOINT_DIR, '*.pt'))))


## 6. Pick the checkpoint epoch


In [ ]:
import re

ckpt_files = glob.glob(os.path.join(CHECKPOINT_DIR, 'weights_epoch_*.pt'))
assert ckpt_files, f'No weights_epoch_*.pt found in {CHECKPOINT_DIR}'

epochs = sorted(int(re.search(r'weights_epoch_(\d+)\.pt$', f).group(1)) for f in ckpt_files)
epoch = POLICY_EPOCH if POLICY_EPOCH is not None else epochs[-1]
assert epoch in epochs, f'weights_epoch_{epoch}.pt not found; available: {epochs}'

CHECKPOINT = f'{OUTPUT_MODEL_NAME}/weights_epoch_{epoch}.pt'
print('Available epochs:', epochs)
print('Probing          :', CHECKPOINT)


## 7. Load the probing code

`scripts/probe/` holds the experiment: `targets.py` (the label registry),
`features.py` (frozen-feature extraction), `fit.py` (the read-outs) and
`run_probing.py` (the same thing as a CLI). Read each module's docstring —
they carry the design decisions this notebook only summarizes.


In [ ]:
import sys

sys.path.insert(0, os.path.join(REPO_ROOT, 'scripts', 'probe'))

import features as ft
import fit as fitting
import targets as tg

print(f'{len(tg.TARGETS)} targets registered:')
for group in tg.GROUPS:
    names = [t.name for t in tg.TARGETS if t.group == group]
    print(f'  {group:<9s} ({len(names)}) {", ".join(names)}')
print()
print('feature variants:', ', '.join(ft.DEFAULT_VARIANTS))


Which targets and which feature variants to run. Everything is the default;
trim `VARIANTS` to `('emb', 'pixels_lowres')` for a fast first look.

| variant | what it is | label read at |
|---|---|---|
| `backbone_cls` | ViT CLS token, **before** the projector | current frame |
| `emb` | projector output — what the LeWM loss and the predictor see | current frame |
| `emb_hist` | `emb` over all `HISTORY_SIZE` context frames | current frame |
| `pred_emb` | the world model's **prediction** of the next latent | predicted frame |
| `emb_next_true` | the true latent of that frame; the ceiling for `pred_emb` | predicted frame |

| `pixels_lowres` | 16×16 average-pooled input frame — the pixel control | current frame |

**One statistical caveat, reported per target as `n_train_effective`.** Seven
labels are constant within an episode — every domain-randomization axis
(`digit_value`, `digit_pos`, `digit_size`, `floor_material`, `wall_material`,
`floor_rgb`, `light_pos`). Sampling 20 windows from one episode gives 20
*identical* labels, so their effective training size is the number of
episodes, not the number of windows: 1,000 rather than 20,000. Their scores
are the noisiest in the table, and they are the targets for which an
episode-level split is not merely good practice but the only split that means
anything.


In [ ]:
PROBE_TARGETS = tg.select_targets()          # or: tg.select_targets(groups=['state'])
VARIANTS = ft.DEFAULT_VARIANTS
LABEL_COLUMNS = tg.required_columns(PROBE_TARGETS)

print(f'{len(PROBE_TARGETS)} targets, {len(VARIANTS)} variants')
print(f'{len(LABEL_COLUMNS)} label columns to load: {LABEL_COLUMNS}')


## 8. Extract frozen features

Runs the encoder once per configuration and caches everything to `.npz`:
every feature variant, every label column at every window step, and the
window provenance (episode + start step). Re-running the cell reuses the
cache — delete the file to force a re-encode.

Two configurations: the trained checkpoint, and an **untrained encoder of the
same architecture**. The second is the control that separates "the
representation encodes this" from "a 384-dim random projection of an image
plus a fitted read-out encodes this".

At the default scale each cache is ~28,000 windows x 3,072 feature dims,
i.e. **~400 MB per configuration** under `$STABLEWM_HOME/probing/`. Trim
`VARIANTS` if that is tight.


In [ ]:
from pathlib import Path


def extract_config(checkpoint, random_init=False):
    return ft.ExtractConfig(
        dataset_name=DATASET_NAME,
        checkpoint=checkpoint,
        random_init=random_init,
        history_size=HISTORY_SIZE,
        num_preds=NUM_PREDS,
        frameskip=FRAMESKIP,
        img_size=IMG_SIZE,
        episodes={'train': TRAIN_EPISODES, 'val': VAL_EPISODES, 'test': TEST_EPISODES},
        windows_per_episode=WINDOWS_PER_EPISODE,
        variants=tuple(VARIANTS),
        seed=SEED,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        dtype=DTYPE,
    )


def get_features(tag, checkpoint, random_init=False):
    cache = Path(RESULTS_ROOT) / f'features_{tag}.npz'
    if cache.exists():
        print(f'Reusing {cache}')
        return ft.load_features(cache)
    payload = ft.extract(extract_config(checkpoint, random_init), LABEL_COLUMNS)
    ft.save_features(cache, payload)
    return payload


RUNS = {
    f'trained_epoch{epoch}': (CHECKPOINT, False),
    'random_init': (CHECKPOINT, True),   # same architecture, no weights
}

payloads = {tag: get_features(tag, ckpt, rnd) for tag, (ckpt, rnd) in RUNS.items()}

meta = payloads[f'trained_epoch{epoch}']['meta']
print()
print('windows per split :', meta['num_windows'])
print('feature dims      :', meta['feature_dims'])
print('label step per variant:', meta['variant_label_step'])


The episode split is disjoint by construction — worth asserting rather than
trusting, since it is the one mistake that would quietly invalidate every
number below.


In [ ]:
eps = meta['episodes']
train_eps, val_eps, test_eps = (set(eps[s]) for s in ('train', 'val', 'test'))
assert not (train_eps & val_eps), 'train/val episode overlap'
assert not (train_eps & test_eps), 'train/test episode overlap'
assert not (val_eps & test_eps), 'val/test episode overlap'
print(f'episodes: {len(train_eps)} train / {len(val_eps)} val / {len(test_eps)} test, pairwise disjoint')

# Same episodes and the same windows across configurations, so the trained
# model and the control are compared on identical data.
for tag, payload in payloads.items():
    assert payload['meta']['episodes'] == eps, f'{tag} used a different split'
    for split in ft.SPLITS:
        assert (payload['windows'][split]['clip_index']
                == payloads[f'trained_epoch{epoch}']['windows'][split]['clip_index']).all()
print('all configurations share the same windows')


## 9. World-model sanity check

Before probing `pred_emb`, confirm the predictor is doing something: cosine
similarity between the predicted latent and the true latent of that frame.
The trained model should sit well above the untrained one (which lands near
0). This is not a probing result — it is the check that `pred_emb` and
`emb_next_true` were not mismatched.

Read the number, though: a cosine at **0.999+ / relative MSE ~1e-3** means
one-step prediction at `frameskip=5` is close to trivial in this latent
space, so `pred_emb` and `emb_next_true` will probe *identically* and the
comparison between them carries no information. That is itself worth
reporting — and the reason to look at a longer horizon (raise `NUM_PREDS`,
which requires a matching training run) if you want the prediction question
to bite.


In [ ]:
for tag, payload in payloads.items():
    fid = fitting.prediction_fidelity(payload)
    if not fid:
        continue
    line = '  '.join(
        f'{s}: cos={v["cosine_mean"]:.3f} relMSE={v["relative_mse"]:.4f}'
        for s, v in fid.items()
    )
    print(f'{tag:<22s} {line}')


## 10. Fit the probes

Three rungs per (variant, target). Regression linear probes are solved in
closed form with a ridge penalty chosen on validation — no optimizer to blame
for a low score. Classification probes and all MLP probes are fitted with
AdamW and early-stopped on validation.

Cost scales as `len(VARIANTS) x len(PROBE_TARGETS) x 3`; at the defaults that
is ~360 fits, a few minutes on a GPU.


In [ ]:
fit_cfg = fitting.FitConfig(
    probes=('baseline', 'linear', 'mlp'),
    mlp_hidden_dim=512,
    mlp_layers=1,
    epochs=200,
    patience=25,
    seed=SEED,
)

rows = []
for tag, payload in payloads.items():
    print(f'=== {tag} ===')
    run_rows, _ = fitting.fit_all(
        payload, PROBE_TARGETS, fit_cfg, variants=VARIANTS, progress=True
    )
    for row in run_rows:
        row['run'] = tag
    rows.extend(run_rows)

print(f'\n{len(rows)} fits done')


In [ ]:
import json

import pandas as pd

df = pd.DataFrame(rows)
df.to_csv(os.path.join(RESULTS_ROOT, f'probe_results_epoch{epoch}.csv'), index=False)
with open(os.path.join(RESULTS_ROOT, f'probe_results_epoch{epoch}.json'), 'w') as f:
    json.dump({'rows': rows, 'manifests': {t: p['meta'] for t, p in payloads.items()}}, f, indent=2)

print('saved to', RESULTS_ROOT)
df.head()


## 11. Results

### 11.1 Score per target × probe, trained model

`R2` for regression, accuracy for classification. Read each row against its
`baseline` column, and each `state` row against the `pixels_lowres` variant
and the `random_init` run further down.


In [ ]:
TRAINED = f'trained_epoch{epoch}'


def score_table(frame, run, probe):
    sub = frame[(frame.run == run) & (frame.probe == probe)]
    table = sub.pivot_table(index='target', columns='variant', values='score')
    order = [t.name for t in PROBE_TARGETS if t.name in table.index]
    cols = [v for v in VARIANTS if v in table.columns]
    table = table.loc[order, cols]
    table.insert(0, 'group', [tg.TARGETS_BY_NAME[t].group for t in table.index])
    table.insert(1, 'metric', [
        'acc' if tg.TARGETS_BY_NAME[t].kind == 'classification' else 'R2'
        for t in table.index
    ])
    return table


linear_scores = score_table(df, TRAINED, 'linear')
linear_scores.round(3)


In [ ]:
mlp_scores = score_table(df, TRAINED, 'mlp')
mlp_scores.round(3)


### 11.2 The three headline comparisons

One frame with the columns the experiment was designed to produce, for the
`emb` variant (the representation the LeWM loss and the planner actually
operate on):

- `baseline` — predict the training mean / majority class.
- `linear`, `mlp` — the two capacity rungs. `mlp - linear` is information
  that is present but **not linearly decodable**.
- `pixels_lowres` — the same probe on a 16×16 image. A `state` target where
  the representation does not beat this has added nothing.
- `random_init` — the same probe on an untrained encoder. The margin over
  this column is what **training** bought.


In [ ]:
def headline(frame, variant='emb'):
    sub = frame[frame.variant == variant]
    wide = sub.pivot_table(index='target', columns=['run', 'probe'], values='score')
    pix = frame[(frame.variant == 'pixels_lowres') & (frame.run == TRAINED)]
    pix = pix.pivot_table(index='target', columns='probe', values='score')

    order = [t.name for t in PROBE_TARGETS if t.name in wide.index]
    out = pd.DataFrame(index=order)
    out['group'] = [tg.TARGETS_BY_NAME[t].group for t in order]
    out['metric'] = ['acc' if tg.TARGETS_BY_NAME[t].kind == 'classification' else 'R2'
                     for t in order]
    out['eff_N'] = [
        int(sub[sub.target == t].n_train_effective.iloc[0]) for t in order
    ]
    out['baseline'] = wide[(TRAINED, 'baseline')].reindex(order)
    out['linear'] = wide[(TRAINED, 'linear')].reindex(order)
    out['mlp'] = wide[(TRAINED, 'mlp')].reindex(order)
    out['mlp - linear'] = out['mlp'] - out['linear']
    if 'linear' in pix.columns:
        out['pixels_linear'] = pix['linear'].reindex(order)
        out['vs pixels'] = out['linear'] - out['pixels_linear']
    if ('random_init', 'linear') in wide.columns:
        out['randinit_linear'] = wide[('random_init', 'linear')].reindex(order)
        out['vs randinit'] = out['linear'] - out['randinit_linear']
    return out


headline(df).round(3)


### 11.3 Regression error in physical units

R² answers "how much of the variance", MAE answers "how wrong, in metres".
Both matter: a cube-position probe at R² = 0.9 with a 2 cm MAE is above the
planner's 4 cm subgoal tolerance (`Run.md` §7) and one at 0.9 with 5 mm is
not.


In [ ]:
reg = df[(df.run == TRAINED) & (df.kind == 'regression') & (df.probe != 'baseline')]
units = {t.name: t.units for t in PROBE_TARGETS}
err = reg.pivot_table(index=['target', 'probe'], columns='variant', values='mae')
err.insert(0, 'units', [units[t] for t, _ in err.index])
err.round(4)


### 11.4 Within-episode ceiling

The episode split asks a hard question: decode the state of an episode whose
lighting, camera angle, cube colours and materials the probe has never seen.
Re-fitting the *same* features on a deliberately leaky **window-level** split
— same episodes on both sides — separates two very different failure modes:

- **high leaky, low episode-split** → the information is in the
  representation, but entangled with episode-level appearance. More training
  episodes, or an appearance-invariant encoder, would help.
- **low both** → the information is not in the representation at all.

This is a diagnostic, not a result: the leaky number is inflated by
construction and must never be quoted on its own. It is meaningless for the
episode-constant targets (a leaky split hands the probe the answer), so they
are excluded.


In [ ]:
import numpy as np


def leaky_reference(payload, probe_targets, variant='emb', seed=0):
    """Refit on a window-level split of the pooled windows (leaky on purpose)."""
    def pool(kind):
        return {
            k: np.concatenate([payload[kind][s][k] for s in ft.SPLITS])
            for k in payload[kind]['train']
        }

    feats, labs = pool('features'), pool('labels')
    n = len(feats[variant])
    perm = np.random.default_rng(seed).permutation(n)
    idx = {
        'train': perm[: int(0.7 * n)],
        'val': perm[int(0.7 * n) : int(0.8 * n)],
        'test': perm[int(0.8 * n) :],
    }
    step = payload['meta']['variant_label_step'][variant]
    cfg = fitting.FitConfig(probes=('baseline', 'linear'), seed=seed)

    out = {}
    for target in probe_targets:
        if target.episode_constant:
            continue
        y_all = tg.build_labels(target, labs, step)
        x = {s: feats[variant][i] for s, i in idx.items()}
        y = {s: y_all[i] for s, i in idx.items()}
        _, score, _, _, _ = fitting.fit_one('linear', x, y, target, cfg, 'cpu')
        out[target.name] = score
    return out


leaky = leaky_reference(payloads[TRAINED], PROBE_TARGETS)
episode_split_scores = df[
    (df.run == TRAINED) & (df.variant == 'emb') & (df.probe == 'linear')
].set_index('target').score

ceiling = pd.DataFrame({
    'episode split': episode_split_scores.reindex(leaky.keys()),
    'leaky (window split)': pd.Series(leaky),
})
ceiling['generalization gap'] = (
    ceiling['leaky (window split)'] - ceiling['episode split']
)
ceiling['group'] = [tg.TARGETS_BY_NAME[t].group for t in ceiling.index]
ceiling.round(3)


### 11.5 Plots


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

for ax, (probe, title) in zip(axes, [('linear', 'Linear probe'), ('mlp', 'MLP probe')]):
    sub = df[(df.run == TRAINED) & (df.probe == probe)]
    table = sub.pivot_table(index='target', columns='variant', values='score')
    order = [t.name for t in PROBE_TARGETS if t.name in table.index][::-1]
    cols = [v for v in VARIANTS if v in table.columns]
    table = table.loc[order, cols]

    y = np.arange(len(order))
    height = 0.8 / max(len(cols), 1)
    for i, col in enumerate(cols):
        ax.barh(y + i * height, table[col].values, height=height, label=col)
    ax.set_yticks(y + 0.4 - height / 2)
    ax.set_yticklabels([
        f'{t} [{tg.TARGETS_BY_NAME[t].group[:4]}]' for t in order
    ])
    ax.axvline(0, color='k', lw=0.8)
    ax.set_xlim(min(-0.15, float(np.nanmin(table.values)) - 0.05), 1.0)
    ax.set_xlabel('R2 (regression) / accuracy (classification)')
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.3)

axes[0].legend(loc='lower right', fontsize=8)
fig.suptitle(f'Probing {TRAINED}: what is decodable from each representation')
fig.tight_layout()
plt.show()


In [ ]:
# Trained vs untrained encoder, linear probe on `emb`. Points near the
# diagonal are things the ViT prior already exposed; the vertical gap is what
# training added.
sub = df[(df.variant == 'emb') & (df.probe == 'linear')]
wide = sub.pivot_table(index='target', columns='run', values='score')
if 'random_init' in wide.columns:
    fig, ax = plt.subplots(figsize=(7, 7))
    colors = {'state': 'tab:blue', 'nuisance': 'tab:orange', 'control': 'tab:red'}
    for group, color in colors.items():
        names = [t.name for t in PROBE_TARGETS
                 if t.group == group and t.name in wide.index]
        if not names:
            continue
        ax.scatter(wide.loc[names, 'random_init'], wide.loc[names, TRAINED],
                   c=color, label=group, s=45)
        for name in names:
            ax.annotate(name, (wide.loc[name, 'random_init'], wide.loc[name, TRAINED]),
                        fontsize=7, xytext=(4, 3), textcoords='offset points')
    lim = [-0.2, 1.05]
    ax.plot(lim, lim, 'k--', lw=0.8)
    ax.set_xlim(lim), ax.set_ylim(lim)
    ax.set_xlabel('untrained encoder (random_init)')
    ax.set_ylabel(f'trained ({TRAINED})')
    ax.set_title('Linear probe on `emb`: what training bought')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()


## 12. Optional: representation quality over training

Same probes, several epochs. Answers whether the linear decodability of the
state tracks the training loss, or saturates early. Extraction is the cost —
one pass per epoch — so this cell reuses caches and is safe to interrupt and
resume.


In [ ]:
RUN_EPOCH_SWEEP = False        # ← set True
SWEEP_EPOCHS = [1, 4, 8, epoch]
SWEEP_TARGETS = tg.select_targets(groups=['state'])
SWEEP_VARIANTS = ('emb',)

if RUN_EPOCH_SWEEP:
    sweep_rows = []
    sweep_cfg = fitting.FitConfig(probes=('baseline', 'linear'), seed=SEED)
    for ep in SWEEP_EPOCHS:
        assert ep in epochs, f'weights_epoch_{ep}.pt is not on this volume'
        tag = f'trained_epoch{ep}'
        payload = get_features(tag, f'{OUTPUT_MODEL_NAME}/weights_epoch_{ep}.pt')
        run_rows, _ = fitting.fit_all(
            payload, SWEEP_TARGETS, sweep_cfg,
            variants=SWEEP_VARIANTS, progress=False,
        )
        for row in run_rows:
            row['run'], row['epoch'] = tag, ep
        sweep_rows.extend(run_rows)
        print(f'epoch {ep}: done')

    sweep = pd.DataFrame(sweep_rows)
    sweep = sweep[sweep.probe == 'linear']
    curve = sweep.pivot_table(index='epoch', columns='target', values='score')

    fig, ax = plt.subplots(figsize=(9, 5))
    curve.plot(marker='o', ax=ax)
    ax.set_xlabel('checkpoint epoch')
    ax.set_ylabel('linear probe R2 / acc on `emb`')
    ax.set_title('Linear decodability of the state over training')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    plt.show()
    display(curve.round(3))


## 13. How to read this

**A high score is a claim about the representation only if the controls are
low.** In order of how often they bite:

1. `score > baseline` — otherwise the probe learned the marginal, nothing else.
2. `score > pixels_lowres` on the same target — otherwise the information was
   already in a 16×16 thumbnail and the encoder is not what exposed it.
3. `score > random_init` — otherwise it is the architecture's prior plus probe
   capacity, not training.
4. The `control` group stays near its baseline — but the four controls are
   not equally strong, so *which* one lights up is the diagnosis:
   - `joint_vel` is the cleanest. Single-frame variants (`emb`,
     `backbone_cls`) must fail on it. `emb_hist` may legitimately succeed —
     three frames 5 steps apart do determine a velocity — so a large
     `emb_hist` − `emb` gap there is evidence the history features work, not
     a leak. `emb` succeeding on it *is* a leak.
   - `target_block_pos` is never rendered and is drawn independently of the
     visible scene. It should sit at baseline.
   - `block_0_pos` is the identity control: positions are visible, the
     index→cube mapping is not, because colours are re-drawn every episode.
     Read it against `cube_pos_sorted`, which asks for the same information
     permutation-invariantly.
   - `target_block` is a **weak** control and is *expected* above baseline:
     the marker is not drawn, but the oracle steers the arm toward it, so
     gripper-vs-cube geometry leaks it. Only a near-perfect score is
     suspicious.

   A genuine leak — an episode in two splits — is ruled out by cell 8's
   assertions, so look instead at the labels: an episode-constant label
   predicted from episode-level appearance is the usual culprit.

**`mlp − linear`** is information present but not linearly decodable. Large
gaps are a real finding, not noise — but check `hp_weight_decay` and
`val_score` before believing a gap smaller than the spread across seeds.

**`pred_emb` vs `emb_next_true`** is the world-model question rather than the
representation question: how much of the *future* state the model's own
prediction carries, against how much the true encoding of that frame carries.
The gap is prediction error; cell 9 quantifies it independently.

**Check `eff_N` before believing a nuisance-target score.** The seven
episode-constant labels have one independent sample per *episode*, so at the
default scale their effective training size is 1,000, not 20,000 — and their
scores are correspondingly noisy. A within-episode target at the same score is
backed by 20× the data.

**Everything reading ~0 usually means too few episodes, not a dead
representation.** Domain randomization makes episode-level appearance the
dominant direction in the features; with ~100 training episodes a linear probe
fits episode identity and every within-episode target lands at R² ≈ 0 on held
out episodes. Section 11.4's leaky reference distinguishes that case from a
representation that genuinely lacks the information.

**Known label noise.** A cube can sit on top of the digit decal (`Run.md` §4)
— `privileged/digit_0_value` still reports the digit in full, so the
`digit_value` ceiling is below 1.0. Filter on `privileged/digit_0_pos` versus
the cube positions to quantify it.

### Same thing from a terminal

```bash
python scripts/probe/run_probing.py \
    --checkpoint lewm_q4_dr/weights_epoch_11.pt \
    --with-random-init \
    --out $STABLEWM_HOME/probing/lewm_q4_dr
```
